# **PROJECT TITLE: Multiclass Fish Image Classification**

#### **Domain: Image Classification**

#### **By: Shubham Pandey**

### **Problem Statement:**
This project focuses on classifying fish images into multiple categories using deep learning models. The task involves training a CNN from scratch and leveraging transfer learning with pre-trained models to enhance performance. The project also includes saving models for later use and deploying a Streamlit application to predict fish categories from user-uploaded images.

#### **Github Link:**
*https://github.com/Shubhampandey1git/Multiclass-Fish-Image-Classification.git*

### **Imports**

In [1]:
# Data LOading and model training
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers

# Visualization
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

# Transfer Learning
from keras.applications import VGG16, ResNet50, MobileNetV2, InceptionV3, EfficientNetB0

import pandas as pd
from keras.models import load_model

In [6]:
# Checking for available GPUs
print('Num of GPUs Available:',tf.config.list_physical_devices('GPU'))

Num of GPUs Available: []


In [7]:
import tensorflow as tf

print(tf.__version__)
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))
print(tf.test.is_built_with_cuda())        # Should be True if GPU-enabled
print(tf.test.is_gpu_available(cuda_only=True, min_cuda_compute_capability=None))  # Deprecated, but helpful


2.16.1
Num GPUs Available:  0
False
Instructions for updating:
Use `tf.config.list_physical_devices('GPU')` instead.
False


In [8]:
import tensorflow as tf
print(tf.sysconfig.get_build_info())


OrderedDict([('is_cuda_build', False), ('is_rocm_build', False), ('is_tensorrt_build', False), ('msvcp_dll_names', 'msvcp140.dll,msvcp140_1.dll')])


In [5]:
import tensorflow as tf
print("TF version:", tf.__version__)
from tensorflow.python.platform import build_info as tf_build_info
print(tf_build_info.build_info.keys())   # See available keys
print(tf_build_info.build_info)          # Print everything


TF version: 2.16.1
odict_keys(['is_cuda_build', 'is_rocm_build', 'is_tensorrt_build', 'msvcp_dll_names'])
OrderedDict([('is_cuda_build', False), ('is_rocm_build', False), ('is_tensorrt_build', False), ('msvcp_dll_names', 'msvcp140.dll,msvcp140_1.dll')])


### **Data Loading, Preprocessing and Augmentaion**

In [ ]:
# Data augmentation for training
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True,
)

val_test_datagen = ImageDataGenerator(rescale=1./255)

# Generators
train_gen = train_datagen.flow_from_directory(
    'data/train',
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical'
)

val_gen = val_test_datagen.flow_from_directory(
    'data/val',
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical'
)

test_gen = val_test_datagen.flow_from_directory(
    'data/test',
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    shuffle=False
)

# Saving the class names for later
class_names = list(train_gen.class_indices.keys())
print("Classes:", class_names)

# **Custom CNN Model Training**

Baseline CNN model

In [ ]:
cnn_model = models.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(224, 224, 3)),
    layers.MaxPooling2D(2, 2),

    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D(2, 2),

    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.MaxPooling2D(2, 2),

    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(len(class_names), activation='softmax')
])


Compiling model

In [ ]:
cnn_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

Training the model

In [ ]:
# Train model
history_cnn = cnn_model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=10  # you can start with 3-5 for a quick test
)

Saving the model

In [ ]:
cnn_model.save('models/cnn_baseline.h5')

### **Visualization**

Accuracy Plot

In [ ]:
plt.figure(figsize=(10,5))
plt.plot(history_cnn.history['accuracy'], label='Train Accuracy')
plt.plot(history_cnn.history['val_accuracy'], label='Validation Accuracy')
plt.title('CNN Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

Loss Plot

In [ ]:
plt.figure(figsize=(10,5))
plt.plot(history_cnn.history['loss'], label='Train Loss')
plt.plot(history_cnn.history['val_loss'], label='Validation Loss')
plt.title('CNN Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()

- as we can observe that the loss goes down and the accuracy goes up, as the epochs goes up.

# **VGG16 Transfer Learning**

In [ ]:
# Loading VGG16 without top classification layers
base_model = VGG16(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)

# Freeze base model layers
for layer in base_model.layers:
    layer.trainable = False

In [ ]:
# Build model
vgg_model = models.Sequential([
    base_model,
    layers.Flatten(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(len(class_names), activation='softmax')
])

# Compile
vgg_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
vgg_model.summary()

In [ ]:
# Train
history_vgg = vgg_model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=10
)

In [ ]:
# Save model
vgg_model.save("models/vgg16_fish.h5")

### **Visualization**

Accuracy

In [ ]:
plt.figure(figsize=(10,5))
plt.plot(history_vgg.history['accuracy'], label='Train Accuracy')
plt.plot(history_vgg.history['val_accuracy'], label='Validation Accuracy')
plt.title('VGG16 Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

Loss

In [ ]:
plt.figure(figsize=(10,5))
plt.plot(history_vgg.history['loss'], label='Train Loss')
plt.plot(history_vgg.history['val_loss'], label='Validation Loss')
plt.title('VGG16 Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()

### **Evaluation on Test Set**

In [ ]:
vgg_test_loss, vgg_test_acc = vgg_model.evaluate(test_gen)
print(f"VGG16 Test Accuracy: {vgg_test_acc*100:.2f}%")

Classification Report & Confusion Matrix

In [ ]:

# Predictions
y_pred = vgg_model.predict(test_gen)
y_pred_classes = np.argmax(y_pred, axis=1)

# True labels
y_true = test_gen.classes

# Report
print(classification_report(y_true, y_pred_classes, target_names=class_names))

# Confusion matrix
cm = confusion_matrix(y_true, y_pred_classes)
print("Confusion Matrix:\n", cm)

### **Fine-Tuning VGG16**

In [ ]:
# Loading base VGG16 again with pretrained weights
base_model = VGG16(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)

In [ ]:
# Unfreezing the last convolutional block
trainable = False
for layer in base_model.layers:
    if layer.name.startswith('block5'):  # Unfreeze only block5 layers
        trainable = True
    layer.trainable = trainable

# Rebuilding model with unfrozen block5
vgg_finetune = models.Sequential([
    base_model,
    layers.Flatten(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(len(class_names), activation='softmax')
])

In [ ]:
# Compiling with a lower learning rate for fine-tuning
vgg_finetune.compile(
    optimizer=optimizers.Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
vgg_finetune.summary()

In [ ]:
# Train
history_vgg_finetune = vgg_finetune.fit(
    train_gen,
    validation_data=val_gen,
    epochs=5  # shorter training for fine-tuning
)

In [ ]:
# Saving fine-tuned model
vgg_finetune.save("models/vgg16_finetuned_fish.h5")

### **Visualization after fine tuning**

Accuracy

In [ ]:
plt.figure(figsize=(10,5))
plt.plot(history_vgg_finetune.history['accuracy'], label='Train Accuracy')
plt.plot(history_vgg_finetune.history['val_accuracy'], label='Validation Accuracy')
plt.title('VGG16 Fine-Tuning Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

Loss

In [ ]:
plt.figure(figsize=(10,5))
plt.plot(history_vgg_finetune.history['loss'], label='Train Loss')
plt.plot(history_vgg_finetune.history['val_loss'], label='Validation Loss')
plt.title('VGG16 Fine-Tuning Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()

### **Evaluation of the fine-tuned model**

In [ ]:
ft_test_loss, ft_test_acc = vgg_finetune.evaluate(test_gen)
print(f"VGG16 Fine-tuned Test Accuracy: {ft_test_acc*100:.2f}%")
print(f"VGG16 Fine-tuned Test Loss: {ft_test_loss:.4f}")

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

# Predictions
y_pred_ft = vgg_finetune.predict(test_gen)
y_pred_ft_classes = np.argmax(y_pred_ft, axis=1)

# True labels
y_true_ft = test_gen.classes

# Report
print("\nClassification Report:")
print(classification_report(y_true_ft, y_pred_ft_classes, target_names=class_names))

# Confusion Matrix
cm_ft = confusion_matrix(y_true_ft, y_pred_ft_classes)
print("Confusion Matrix:\n", cm_ft)


### **Saving the model**

In [ ]:
vgg_finetune.save("models/vgg16_finetuned_fish.h5")

# **Creating a function for Transfer Learning & Fine-Tuning**

In [ ]:
def train_and_finetune_model(model_name, base_model_class, img_size=(224,224), unfreeze_from_layer=None):
    print(f"\n=== Training {model_name} ===")

    # Load base model
    base_model = base_model_class(
        weights='imagenet',
        include_top=False,
        input_shape=(img_size[0], img_size[1], 3)
    )

    # Freeze all layers initially
    for layer in base_model.layers:
        layer.trainable = False

    # Build model
    model = models.Sequential([
        base_model,
        layers.Flatten(),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(len(class_names), activation='softmax')
    ])

    # Compile
    model.compile(
        optimizer='adam',
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    # Train base model
    history = model.fit(
        train_gen,
        validation_data=val_gen,
        epochs=5
    )

    # Fine-tune from specific layer if provided
    if unfreeze_from_layer:
        trainable = False
        for layer in base_model.layers:
            if layer.name.startswith(unfreeze_from_layer):
                trainable = True
            layer.trainable = trainable

        model.compile(
            optimizer=optimizers.Adam(learning_rate=1e-5),
            loss='categorical_crossentropy',
            metrics=['accuracy']
        )

        history_ft = model.fit(
            train_gen,
            validation_data=val_gen,
            epochs=3
        )
    else:
        history_ft = None

    # Save model
    model.save(f"models/{model_name.lower()}_finetuned.h5")

    return history, history_ft, model

# **Calling the fun for each model**

### **ResNet50**

In [ ]:
hist_resnet, hist_resnet_ft, resnet_model = train_and_finetune_model(
    "ResNet50", ResNet50, img_size=(224,224), unfreeze_from_layer="conv5"
)

### **MobileNetV2**

In [ ]:
hist_mobile, hist_mobile_ft, mobile_model = train_and_finetune_model(
    "MobileNetV2", MobileNetV2, img_size=(224,224), unfreeze_from_layer="block_15"
)

### **InceptionV3**

In [ ]:
hist_incep, hist_incep_ft, incep_model = train_and_finetune_model(
    "InceptionV3", InceptionV3, img_size=(299,299), unfreeze_from_layer="mixed9"
)

### **EfficientNetB0**

In [ ]:
hist_effnet, hist_effnet_ft, effnet_model = train_and_finetune_model(
    "EfficientNetB0", EfficientNetB0, img_size=(224,224), unfreeze_from_layer="block6"
)

# **Evaluation function for all models**

In [ ]:
def evaluate_model(model, name):
    # Predictions
    preds = model.predict(test_gen)
    y_pred = np.argmax(preds, axis=1)
    y_true = test_gen.classes

    # Metrics
    acc = (y_pred == y_true).mean()
    prec = precision_score(y_true, y_pred, average='weighted', zero_division=0)
    rec = recall_score(y_true, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)

    return {
        "Model": name,
        "Accuracy": round(acc*100, 2),
        "Precision": round(prec*100, 2),
        "Recall": round(rec*100, 2),
        "F1-score": round(f1*100, 2)
    }


### **Evaluations for all the models**

In [ ]:
model_list = [
    ("Custom CNN", "models/cnn_baseline.h5"),
    ("VGG16 Fine-tuned", "models/vgg16_finetuned_fish.h5"),
    ("ResNet50 Fine-tuned", "models/resnet50_finetuned.h5"),
    ("MobileNetV2 Fine-tuned", "models/mobilenetv2_finetuned.h5"),
    ("InceptionV3 Fine-tuned", "models/inceptionv3_finetuned.h5"),
    ("EfficientNetB0 Fine-tuned", "models/efficientnetb0_finetuned.h5")
]

In [ ]:
results = []
for name, path in model_list:
    print(f"Evaluating {name}...")
    model = load_model(path)
    results.append(evaluate_model(model, name))


In [ ]:
df_results = pd.DataFrame(results)
print("\nFinal Model Comparison:\n")
print(df_results.to_string(index=False))